# **CLASSIFICATION MODEL NOTEBOOK**

## Objectives

* Fit and evaluate a classification model to predict if a hotel booking will be cancelled or not.

## Inputs

* **Raw Dataset:** inputs/datasets/raw/hotel_bookings.csv
* **Data cleaning pipeline** from notebook 4
* Suggested **feature engineering pipeline** from notebook 5

## Outputs

* **Train set** (features and target)
* **Test set** (features and target)
* Finalised **data cleaning and feature engineering pipelines**
* **Modeling pipeline**
* Feature importance plot

---

# Imports

General imports needed for models and preparing data

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pipelines and custom transformers
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Data cleaning and feature engineering pipelines 
from feature_engine.imputation import CategoricalImputer, ArbitraryNumberImputer, MeanMedianImputer
from feature_engine.outliers import ArbitraryOutlierCapper
from feature_engine.encoding import OrdinalEncoder, OneHotEncoder, RareLabelEncoder
from feature_engine.creation import CyclicalFeatures, MathFeatures
from feature_engine.transformation import YeoJohnsonTransformer, PowerTransformer
from feature_engine.selection import SmartCorrelatedSelection

# ML Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

# ML algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier

# Load, Pre-Process and Split Data

## Load Data

Load the raw dataset

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

## Pre-Split Data Cleaning

***NOTE:*** *These steps must be done at this stage to avoid data leaking into the test set.*

Drop columns `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Aggregate duplicates into single records

In [ ]:
df = df.value_counts(dropna=False).reset_index(name='record_count')
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

# Define ML Pipelines

Use Data Cleaning pipeline from notebook 4.

Initially, we will use the Feature Engineering pipeline suggested in notebook 5. 

## Data Cleaning Pipeline

Define custom transformer for imputing `is_repeated_guest`

In [ ]:
class RepeatedGuestImputer(BaseEstimator, TransformerMixin):
    def __init__(self, booking_col='previous_bookings_not_canceled', target_col='is_repeated_guest'):
        self.booking_col = booking_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Set a dummy attribute to signal fitted status (to remove warning)
        self.fitted_ = True
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isnull()
        X.loc[mask_missing, self.target_col] = (X.loc[mask_missing, self.booking_col] > 0).astype(int)
        return X


Define variables and transformers for using in the pipeline

In [ ]:
# Categorical imputer for 'Missing' label
impute_missing_label_variables = ['country', 'company', 'agent']
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=impute_missing_label_variables,
)

# Categorical imputer for mode
impute_mode_variables = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]
imputer_mode = CategoricalImputer(
    imputation_method='frequent',
    variables=impute_mode_variables
)

# Categorical imputer for repeated guest
imputer_repeated_guest = RepeatedGuestImputer()

# Numeric imputer for zero
impute_zero_variables = [
    'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=impute_zero_variables
)

# Numeric imputer for median
impute_median_variables = ['adr', 'adults']
imputer_median = MeanMedianImputer(
    imputation_method='median',
    variables=impute_median_variables
)

# Outlier capper
max_capping_dict = {
    'lead_time': 600,
    'arrival_date_year': 2017,
    'arrival_date_week_number': 53,
    'arrival_date_day_of_month': 31,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'adults': 4,
    'children': 4,
    'babies': 2,
    'previous_cancellations': 10,
    'previous_bookings_not_canceled': 20,
    'booking_changes': 10,
    'days_in_waiting_list': 60,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_year': 2015,
    'arrival_date_week_number': 1,
    'arrival_date_day_of_month': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'adults': 1,
    'children': 0,
    'babies': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'days_in_waiting_list': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Define data cleaning pipeline

In [ ]:
data_cleaning_pipeline = Pipeline([
    ('imputer_missing_label', imputer_missing_label),
    ('imputer_mode', imputer_mode),
    ('imputer_repeated_guest', imputer_repeated_guest),
    ('imputer_zero', imputer_zero),
    ('imputer_median', imputer_median),
    ('outlier_capper', outlier_capper),
])

## Feature Engineering Pipeline

Define custom transformers

In [ ]:
class CancellationRatio(BaseEstimator, TransformerMixin):
    """
    Creates cancellation_ratio = previous_cancellations / (previous_total_bookings)
    """

    def __init__(self,
                 cancel_col='previous_cancellations',
                 no_cancel_col='previous_bookings_not_canceled',
                 new_col_name='cancellation_ratio'):
        self.cancel_col = cancel_col
        self.no_cancel_col = no_cancel_col
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        total_prev = X[self.cancel_col] + X[self.no_cancel_col]
        # Avoid division by zero
        X[self.new_col_name] = np.where(total_prev == 0, 0, X[self.cancel_col] / total_prev)
        return X


class MonthMapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables):
        self.variables = variables
        self.month_map = {
            'January': 1, 'February': 2, 'March': 3, 'April': 4,
            'May': 5, 'June': 6, 'July': 7, 'August': 8,
            'September': 9, 'October': 10, 'November': 11, 'December': 12
        }
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.month_map)
        return X

Define variables and transformers

In [ ]:
# Ordinal Encoder
binary_ordinal_encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=['hotel', 'is_repeated_guest']
)

# Month Mapper (before cyclical encoding)
month_mapper = MonthMapper(variables=['arrival_date_month'])

# Rare Label Encoders (before one hot encoding)
rare_country_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['country']
)
rare_agent_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['agent']
)
rare_company_encoder = RareLabelEncoder(
    tol=0.01,
    n_categories=1,
    variables=['company']
)

# One-Hot Encoder (after rare label encoders)
one_hot_encoder = OneHotEncoder(
    variables=[
        'meal', 'market_segment', 'distribution_channel', 'reserved_room_type',
        'assigned_room_type', 'deposit_type', 'customer_type', 'country', 'agent', 'company'],
    drop_last=False
)

# Cyclical Encoders (after Month Mapper)
cyclical_features = CyclicalFeatures(
    variables=['arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month'],
    drop_original=True
)

# Create New Features
create_total_stays = MathFeatures(
    variables=['stays_in_weekend_nights', 'stays_in_week_nights'],
    func='sum',
    new_variables_names=['total_stay_length']
)
create_total_cost = MathFeatures(
    variables=['adr', 'total_stay_length'],
    func='prod',
    new_variables_names=['total_cost']
)
create_cancellation_ratio = CancellationRatio()  # requires 'previous_cancellations' and 'previous_bookings_not_canceled'

# Numeric Transformers
adr_transformer = YeoJohnsonTransformer(variables=['adr'])
lead_time_transformer = PowerTransformer(variables=['lead_time'])

# Smart Correlated Selection
smart_corr_sel = SmartCorrelatedSelection(
    variables=None,
    method='spearman',
    threshold=0.6,
    selection_method='variance'
)

Define feature engineering pipeline

In [ ]:
feature_engineering_pipeline = Pipeline([
    ('binary_ordinal_encoder', binary_ordinal_encoder),
    ('month_mapper', month_mapper),
    ('rare_country_encoder', rare_country_encoder),
    ('rare_agent_encoder', rare_agent_encoder),
    ('rare_company_encoder', rare_company_encoder),
    ('one_hot_encoder', one_hot_encoder),
    ('cyclical_features', cyclical_features),
    ('create_total_stays', create_total_stays),
    ('create_total_cost', create_total_cost),
    ('create_cancellation_ratio', create_cancellation_ratio),
    ('adr_transformer', adr_transformer),
    ('lead_time_transformer', lead_time_transformer),
    ('smart_corr_sel', smart_corr_sel),
])

## PipelineDataCleaningAndFeatureEngineering

Combine the previous two pipelines into a single pipeline.

In [ ]:
def PipelineDataCleaningAndFeatureEngineering():
    pipeline_base = Pipeline([
        ("data_cleaning_pipeline", data_cleaning_pipeline),
        ("feature_engineering_pipeline", feature_engineering_pipeline),
    ])

    return pipeline_base

PipelineDataCleaningAndFeatureEngineering()

## Classification Pipeline

In [ ]:
def PipelineClf(model):
    pipeline_base = Pipeline([
        ("scaler", StandardScaler()),
        ("feat_selection", SelectFromModel(model)),
        ("model", model),
    ])

    return pipeline_base


# Helper Functions

Helper for finding best model and hyperparameters (written by Code Institute)

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineClf(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

Adapted from a function written by Code Institute
- Includes extra summary tables to more easily compare the model performance on testing and training data.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def target_class_summary(X_train, y_train, X_test, y_test, pipeline, label_map, target_class):

    # Model performance on training dataset
    y = y_train
    prediction = pipeline.predict(X_train)

    train_report = classification_report(
        y, prediction, target_names=label_map, output_dict=True
    )
    train_precision = train_report[target_class]["precision"]
    train_recall = train_report[target_class]["recall"]
    train_f1 = train_report[target_class]["f1-score"]

    # Model performance on testing dataset
    y = y_test
    prediction = pipeline.predict(X_test)

    test_report = classification_report(
        y, prediction, target_names=label_map, output_dict=True
    )
    test_precision = test_report[target_class]["precision"]
    test_recall = test_report[target_class]["recall"]
    test_f1 = test_report[target_class]["f1-score"]

    results = {
        "Dataset": ['Train', 'Test'],
        "Precision": [f"{train_precision:.2f}", f"{test_precision:.2f} ({(test_precision - train_precision):.2f})"],
        "Recall": [f"{train_recall:.2f}", f"{test_recall:.2f} ({(test_recall - train_recall):.2f})"],
        "F1-Score": [f"{train_f1:.2f}", f"{test_f1:.2f} ({(test_f1 - train_f1):.2f})"],
    }

    overview =  pd.DataFrame(results).set_index('Dataset')
    overview.index.name = None
    display(overview)


def confusion_matrix_and_report(X, y, pipeline, label_map):
    prediction = pipeline.predict(X)

    print("---  Confusion Matrix  ---")
    cm = confusion_matrix(y_true=y, y_pred=prediction)
    print(
        pd.DataFrame(
            cm,
            index=["Actual " + sub for sub in label_map],
            columns=["Predicted " + sub for sub in label_map],
        )
    )
    print("\n")

    print("---  Classification Report  ---")
    print(classification_report(y, prediction, target_names=label_map), "\n")


def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map, target_class=None, summary_only=False):
    # Show Summary
    if target_class:
        print(f'#### Summary on "{target_class}" class ####')
        # print('Comparison with performance on training data is shown in brackets')
        target_class_summary(X_train, y_train, X_test, y_test, pipeline, label_map, target_class)
        
    else:
        print('No target class specified\n')
    
    # Show Drilldown
    if not summary_only:
        print("#### Train Set #### \n")
        confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

        print("#### Test Set ####\n")
        confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

# Target Imbalance

## Approaches for Handling Target Imbalance

Apply `PipelineDataCleaningAndFeatureEngineering` by:
- Fitting pipeline to train dataset
- Transforming both train and test datasets

In [ ]:
pipeline_data_cleaning_feat_eng = PipelineDataCleaningAndFeatureEngineering()
X_train = pipeline_data_cleaning_feat_eng.fit_transform(X_train, y_train)
X_test = pipeline_data_cleaning_feat_eng.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

View distribution of target values in train set

In [ ]:
y_train.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

We will test the performance of the various models using 3 different approaches:
1. Keep Imbalance
2. Oversample (SMOTE)
3. Undersample

Next we will prepare edited training data to use for each approach.

## Prepare Data for Each Approach

### 1. Keep Imbalance

We can just use X_train and y_train unedited for this test.

**NO ACTION REQUIRED**

### 2. OverSample (SMOTE)

Fit SMOTE to training data and save oversampled data as `X_train_over` and `y_train_over`

In [ ]:
from imblearn.over_sampling import SMOTE

oversample = SMOTE(sampling_strategy='minority', random_state=0)
X_train_over, y_train_over = oversample.fit_resample(X_train, y_train)

print('Observations in original data:', X_train.shape)
print('Observations in oversampled data:', X_train_over.shape)

Check target distribution of oversampled data

In [ ]:
y_train_over.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

### 3. Undersample

Fit undersampler to training data and save undersampled data as `X_train_under` and `y_train_under`

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

undersample = RandomUnderSampler(sampling_strategy='majority', random_state=0)
X_train_under, y_train_under = undersample.fit_resample(X_train, y_train)

print('Observations in original data:', X_train.shape)
print('Observations in undersampled data:', X_train_over.shape)

Check target distribution of undersampled data

In [ ]:
y_train_under.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

# Find Best Algorithm & Target Imbalance Approach

## Define Models and Scoring Metric

Define all relevant scoring metrics that can be used when evaluating model performance.

Since we are prioritising a recall above 0.8, we will use recall_scorer initially.

In [ ]:
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

recall_scorer = make_scorer(recall_score, pos_label=1)
precision_scorer = make_scorer(precision_score, pos_label=1)
f1_scorer = make_scorer(f1_score, pos_label=1)

Define classification algorithms and hyperparameters to use in initial search.

***NOTE:*** *We are using the default hyperparameters initially*

In [ ]:
models_quick_search = {
    "LogisticRegression": LogisticRegression(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=0),
    "ExtraTreesClassifier": ExtraTreesClassifier(random_state=0),
    "AdaBoostClassifier": AdaBoostClassifier(random_state=0),
}

params_quick_search = {
    "LogisticRegression": {},
    "XGBClassifier": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "GradientBoostingClassifier": {},
    "ExtraTreesClassifier": {},
    "AdaBoostClassifier": {},
}

## Approach 1: Train Models on Imbalanced Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

for i in range(5):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map= ['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Approach 2: Train Models on Oversampled Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train_over, y_train_over, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

for i in range(5):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train_over, y_train=y_train_over,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map= ['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Approach 3: Train Models on Undersampled Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train_under, y_train_under, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

for i in range(5):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train_under, y_train=y_train_under,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map= ['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Analysis of Approaches

When considering the best performing models for each dataset (by 'Cancel' category), here are the findings:
- **Unbalanced:**
  - lowest recall (~0.63) but highest precision (~0.73) - expected since 'Cancel' category is underrepresented
  - F1 scores ~0.68
  - `XGBClassifier` had a strongest performance
  - `ExtraTreesClassifier` and `RandomForestClassifier` performed well but overfitted the training data

- **SMOTE:**
  - recall was slightly higher (~0.68) but precision was much lower (~0.59)
  - F1 scores ~0.62
  - `GradientBoostingClassifier` and `AdaBoostClassifier` performed well

- **Under-Sample:**
  - recall was higher (~0.86) but precision was lower (~0.58)
  - F1 scores ~0.69
  - `XGBClassifier` and `GradientBoostingClassifier` performed well
  - `ExtraTreesClassifier` performed well but overfitted the training data

**CONCLUSIONS**
- Oversampling with SMOTE had a negative impact on the F1 score (due to the lower precision) so will not be used.
- `XGBClassifier` had the strongest performance on the imbalanced and undersampled datasets so this is a good candidate for conducting hyperparameter optimisation.

## Approach 4: Train `XGBClassifier` on Imbalanced Data using `scale_pos_weight`

The `XGBClassifier` model has a parameter called `scale_pos_weight` which allows us to specify the weight that should be given to the 'Cancel' class. Therefore, we can also try this approach.

A typical starting value to try is calculated by sum(negative instances) / sum(positive instances). We will also test values slightly higher and lower than this value.

In [ ]:
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()
scale_pos_weight = num_negative / num_positive

print('Negative instances:', num_negative)
print('Positive instances:', num_positive)
print('scale_pos_weight:', scale_pos_weight)


Define model and parameters

In [ ]:
models_quick_search = {
    "XGBClassifier": XGBClassifier(random_state=0),
}

params_quick_search = {
    "XGBClassifier": {
        'model__scale_pos_weight': [
            scale_pos_weight,
            scale_pos_weight*0.9,
            scale_pos_weight*1.1,
        ],
    },
}

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- This time we will use precision as the performance metric to compare the effect of each parameter value on the precision
- We want a precision above 0.6 if possible

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(5)

Evaluate model performance

In [ ]:
clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map= ['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only=False
                )

## Conclusion and Next Steps

Approach 4 gave the strongest performance of all of the approaches tested.

A hyperparameter optimisation search will now be conducted using `XGBClassifier`.